In [1]:
import requests
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

df = pd.read_csv("webtoon_raw.csv")
title_ids = df["titleId"].tolist()
print(len(title_ids))

test_ids = title_ids[:5]
print(test_ids)

4103
[758037, 844058, 822657, 840014, 832703]


In [2]:
def collect_detail(tid):
    """titleId 하나를 받아 상세 정보를 딕셔너리로 반환"""
    result = {"titleId": tid}
    
    # ① info: 소개글, 태그, 관심수
    url_info = f"https://comic.naver.com/api/article/list/info?titleId={tid}"
    res = requests.get(url_info, headers=headers)
    info = res.json()
    result["synopsis"] = info.get("synopsis")
    result["favoriteCount"] = info.get("favoriteCount")
    tags = info.get("curationTagList", [])
    result["tags"] = ", ".join([t["tagName"] for t in tags])
    
    # ② list: 1화 날짜 (오름차순 정렬로 첫 항목 = 1화)
    url_list = f"https://comic.naver.com/api/article/list?titleId={tid}&page=1&sort=ASC"
    res = requests.get(url_list, headers=headers)
    data = res.json()
    articles = data.get("articleList", [])
    result["firstDate"] = articles[0].get("serviceDateDescription") if articles else None
    result["totalCount"] = data.get("totalCount")
    
    return result


In [4]:
results = []
errors = []

for tid in test_ids:
    try:
        r = collect_detail(tid)
        results.append(r)
        print("성공:", r["titleId"], r.get("firstDate"), r.get("tags"))
    except Exception as e:
        errors.append({"titleId": tid, "error": str(e)})
        print("실패:", tid, "→", e)
    time.sleep(0.3)

print(f"\n성공 {len(results)}개 / 실패 {len(errors)}개")

성공: 758037 20.11.01 액션, 사이다, 학원액션, 자극적인, 사회고발, 참교육, 블루스트링, 학원물, 다크히어로, 먼치킨
실패: 844058 → 'str' object has no attribute 'get'
성공: 822657 24.03.17 무협/사극, 사이다, 동양, 환생, 액션, 고인물, 환골탈태, 동양풍판타지, 판무, 먼치킨, 소설원작
성공: 840014 25.06.29 판타지, 오컬트판타지, 자본주의, 인생역전
성공: 832703 24.09.29 무협/사극, 정통무협, 시리어스, 동양, 모험, 고인물, 힘숨찐, 복수극, 먼치킨, 소설원작

성공 4개 / 실패 1개


In [5]:
results = []
errors = []

for i, tid in enumerate(title_ids):
    try:
        results.append(collect_detail(tid))
    except Exception as e:
        errors.append({"titleId": tid, "error": str(e)})
    
    # 진행상황 표시 + 중간 저장 (200개마다)
    if (i + 1) % 200 == 0:
        pd.DataFrame(results).to_csv("webtoon_detail_temp.csv", index=False, encoding="utf-8-sig")
        print(f"{i+1}/{len(title_ids)} 완료 | 성공 {len(results)} 실패 {len(errors)}")
    
    time.sleep(0.3)

print(f"\n최종: 성공 {len(results)}개 / 실패 {len(errors)}개")

200/4103 완료 | 성공 187 실패 13
400/4103 완료 | 성공 379 실패 21
600/4103 완료 | 성공 563 실패 37
800/4103 완료 | 성공 742 실패 58
1000/4103 완료 | 성공 899 실패 101
1200/4103 완료 | 성공 1086 실패 114
1400/4103 완료 | 성공 1275 실패 125
1600/4103 완료 | 성공 1462 실패 138
1800/4103 완료 | 성공 1649 실패 151
2000/4103 완료 | 성공 1843 실패 157
2200/4103 완료 | 성공 2038 실패 162
2400/4103 완료 | 성공 2227 실패 173
2600/4103 완료 | 성공 2422 실패 178
2800/4103 완료 | 성공 2619 실패 181
3000/4103 완료 | 성공 2811 실패 189
3200/4103 완료 | 성공 3005 실패 195
3400/4103 완료 | 성공 3199 실패 201
3600/4103 완료 | 성공 3391 실패 209
3800/4103 완료 | 성공 3582 실패 218
4000/4103 완료 | 성공 3774 실패 226

최종: 성공 3872개 / 실패 231개


In [6]:
detail_df = pd.DataFrame(results)
detail_df.to_csv("webtoon_detail.csv", index=False, encoding="utf-8-sig")

errors_df = pd.DataFrame(errors)
errors_df.to_csv("webtoon_errors.csv", index=False, encoding="utf-8-sig")

print("성공:", detail_df.shape)
print("실패:", len(errors_df))
detail_df.head()

성공: (3872, 6)
실패: 231


,titleId,synopsis,favoriteCount,tags,firstDate,totalCount
0,758037,무너진 교권을 지키기 위해 교권보호국 소속 나화진의 참교육이 시작된다!\n<부활남>...,1061667,"액션, 사이다, 학원액션, 자극적인, 사회고발, 참교육, 블루스트링, 학원물, 다크...",20.11.01,257
1,822657,"철혈의 맹주, 강호의 절대자 '천하진'. 가문의 수치라 불리는 망나니 '벽리단'의 ...",263765,"무협/사극, 사이다, 동양, 환생, 액션, 고인물, 환골탈태, 동양풍판타지, 판무,...",24.03.17,131
2,840014,“진짜 일류 무당은 여의도에서 논다.” \n주가의 오르내림을 예측하는 것만으로 수십...,376199,"판타지, 오컬트판타지, 자본주의, 인생역전",25.06.29,59
3,832703,'스물을 넘기지 못하고 단명할 운명'을 타고난 정연신.\n집안사람들 모두 연신을 홀...,231488,"무협/사극, 정통무협, 시리어스, 동양, 모험, 고인물, 힘숨찐, 복수극, 먼치킨,...",24.09.29,86
4,796075,단전이 부숴졌다는 이유로 집에서는 내놓은 자식 취급을 받던 소운휘는 혈교에 납치되어...,314839,"무협/사극, 동양, 이능력, 모험, 액션, 성장물, 환골탈태, 회귀, 판무, 복수극...",22.06.19,193


In [7]:
# 각 핵심 컬럼이 얼마나 채워졌는지 (결측 개수)
print(detail_df[["synopsis", "tags", "firstDate", "favoriteCount"]].isna().sum())

# 태그에 '먼치킨'이 들어간 작품 수 (우리 분석의 주인공!)
munchkin = detail_df["tags"].str.contains("먼치킨", na=False).sum()
print(f"\n먼치킨 태그 작품: {munchkin}개")

# firstDate 샘플 확인
print(detail_df["firstDate"].head(10).tolist())

synopsis         0
tags             0
firstDate        0
favoriteCount    0
dtype: int64

먼치킨 태그 작품: 282개
['20.11.01', '24.03.17', '25.06.29', '24.09.29', '22.06.19', '19.09.01', '24.03.10', '21.08.29', '15.05.17', '25.12.03']


In [8]:
# ① 최종 성공/실패 규모
print("성공:", len(detail_df), "/ 실패:", len(errors))

# ② 실패 원인 분포 (성인물이 맞는지)
errors_df = pd.read_csv("webtoon_errors.csv")
print(errors_df["error"].value_counts())

# ③ 성공 데이터에 성인물이 얼마나 섞였는지 교차 확인
raw = pd.read_csv("webtoon_raw.csv")
merged_check = raw[raw["titleId"].isin(errors_df["titleId"])]
print("\n실패 작품 중 성인물(adult=True) 수:", merged_check["adult"].sum(), "/", len(merged_check))

성공: 3872 / 실패: 231
error
'str' object has no attribute 'get'    231
Name: count, dtype: int64

실패 작품 중 성인물(adult=True) 수: 231 / 231


In [9]:
check = pd.read_csv("webtoon_detail.csv")
print(check.shape)
check.head(3)

(3872, 6)


,titleId,synopsis,favoriteCount,tags,firstDate,totalCount
0,758037,무너진 교권을 지키기 위해 교권보호국 소속 나화진의 참교육이 시작된다!\n<부활남>...,1061667,"액션, 사이다, 학원액션, 자극적인, 사회고발, 참교육, 블루스트링, 학원물, 다크...",20.11.01,257
1,822657,"철혈의 맹주, 강호의 절대자 '천하진'. 가문의 수치라 불리는 망나니 '벽리단'의 ...",263765,"무협/사극, 사이다, 동양, 환생, 액션, 고인물, 환골탈태, 동양풍판타지, 판무,...",24.03.17,131
2,840014,“진짜 일류 무당은 여의도에서 논다.” \n주가의 오르내림을 예측하는 것만으로 수십...,376199,"판타지, 오컬트판타지, 자본주의, 인생역전",25.06.29,59
